# Analysis of scored data 



In [ ]:
# Analysis setup


# Future annotations
from __future__ import annotations

import logging as logger
import matplotlib.pyplot as plt
# Third-party
import pandas as pd
import seaborn as sns
import sys
import tqdm
from glob import glob
from itables import init_notebook_mode
# Standard library
from pathlib import Path
from rdkit.Chem import Crippen
from rdkit.Chem import PandasTools
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem.rdMolDescriptors import GetMorganFingerprintAsBitVect

sys.path.append("../binana/python")

init_notebook_mode(all_interactive=True)
# ---------------------------------------------------------------------------

# Optional/visualization extras (guarded usage later in the notebook)
try:
    import py3Dmol  # noqa: F401
except Exception:
    py3Dmol = None

# Notebook/UI init
init_notebook_mode(all_interactive=True)  # itables
IPythonConsole.ipython_useSVG = True  # RDKit nicer SVG rendering

# Plotting defaults
sns.set(context="notebook", style="whitegrid")
plt.rcParams.update(
    {
        "figure.figsize": (7, 4),
        "axes.titlesize": 12,
        "axes.labelsize": 11,
        "legend.fontsize": 9,
    }
)

# Logging
logger.basicConfig(level=logger.INFO, format="%(levelname)s:%(name)s:%(message)s")

# Constants
SDF_RECORD_DELIM = "$$$$"


## load SDF and CSV Files 

In [ ]:

# Create and read Structures to a Dataframe
global_sdf_df = pd.DataFrame()
# outputResults = pd.read_csv("pleaseWork/justmol2mol_4_Compounds/graph-dock/node-reinvent/justmol2mol_4__1.csv", header=0)

sdfFiles = "/home/a/REINVENT4/ReinventStudies/scoringData/sampledDockingData/justmol2mol_4_compounds_Mols"
# outputResults.head()

i = 0
for file in tqdm.tqdm_notebook(glob(f"{sdfFiles}*.sdf")):
    print(i, file)
    df = PandasTools.LoadSDF(file, idName='ID', molColName='ROMol', includeFingerprints=True, isomericSmiles=True,
                             smilesName=None, embedProps=True, removeHs=False, strictParsing=True, sanitize=True)
    # sdf_df = sdf_to_csv(file)
    df['filename'] = [Path(file).stem] * len(df)

    global_sdf_df = pd.concat([global_sdf_df, df], ignore_index=True)
    i = i + 1

global_sdf_df = global_sdf_df.sort_values("filename")
print(f"global_sdf_df Length{len(global_sdf_df)}")
print(f"global_sdf_df Unique Smiles{len(global_sdf_df.smiles.unique())}")

global_sdf_df.head(5)

In [ ]:
outputResults = pd.read_csv(
    "/home/a/REINVENT4/ReinventStudies/scoringData/sampledDockingData/scoredData.csv",
    header=0,
)

print(f"outputResults Length{len(outputResults)}")
print(f"outputResults Unique SMILES{len(outputResults.SMILES.unique())}")
print(f"outputResults Unique SMILES RDKit Length{len(outputResults['RDKit_SMILES (REINVENT)'].unique())}")

outputResults.head()

outputResults["DockingScore (raw)"] = outputResults["DockingScore (raw)"].astype(float)

global_sdf_df["m_score__min__energy"] = global_sdf_df["m_score__min__energy"].astype(
    float
)

outputResults = pd.merge(
    outputResults,
    global_sdf_df,
    # left_on=["RDKit_SMILES (REINVENT)", "DockingScore (raw)"],
    left_on=["SMILES", "DockingScore (raw)"],

    right_on=["smiles", "m_score__min__energy"],
    how="outer",
    validate="one_to_many",
)

outputResults["energy"] = outputResults["energy"].astype(float)
outputResults["rmsd"] = outputResults["rmsd"].astype(float)
print(f"outputResults Length{len(outputResults)}")
print(f"outputResults Unique SMILES{len(outputResults.SMILES.unique())}")
print(f"outputResults Unique SMILES RDKit Length{len(outputResults['RDKit_SMILES (REINVENT)'].unique())}")
outputResults.head()


### add References to enable easy filtering

In [ ]:

from rdkit.Chem.Draw import *


# Python


def slogp_from_smiles(smiles: str) -> float:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Invalid SMILES")
    return Crippen.MolLogP(mol)  # RDKit SlogP (Crippen)


refMols = []

with open("../mols/Rad51/cam833_compounds.smi", "r") as f:
    for line in f:
        refMols.append(Chem.MolFromSmiles(line))

img = MolsToGridImage(
    refMols,
    subImgSize=(400, 400),
    molsPerRow=7,
    useSVG=False,
    returnPNG=True
)
img


# Example


In [ ]:

quantile_10 = outputResults["energy"].quantile(0.1)

outputResults = outputResults.sort_values(["energy", "rmsd"], )
outputResults = outputResults.loc[outputResults['energy'] <= quantile_10]

outputResults = outputResults.dropna(subset=["SMILES", "Score", "DockingScore (raw)"])
outputResults['SlogP'] = outputResults['SMILES'].apply(slogp_from_smiles)

outputResults = outputResults.loc[outputResults['SlogP'] <= 5]
outputResults['mol'] = outputResults['SMILES'].apply(lambda x: Chem.MolFromSmiles(x))

outputResults["morgan"] = outputResults["ROMol"].apply(
    lambda mol: GetMorganFingerprintAsBitVect(mol, 2, nBits=2048) if mol is not None and hasattr(mol,
                                                                                                 'GetNumAtoms') else None
)

print(f"outputResults Length{len(outputResults)}")
print(f"outputResults Unique SMILES{len(outputResults.SMILES.unique())}")
print(f"outputResults Unique SMILES RDKit Length{len(outputResults['RDKit_SMILES (REINVENT)'].unique())}")
outputResults

In [ ]:
from rdkit.Chem.Draw import MolsToGridImage

uniqueStructs = outputResults.drop_duplicates(subset=['SMILES'])

img = MolsToGridImage(
    uniqueStructs['mol'],
    subImgSize=(400, 400),
    molsPerRow=10,
    maxMols=16,
    legends=[f"Docking Score: {row['DockingScore (raw)']}" for _, row in uniqueStructs.iterrows()], useSVG=False,
    returnPNG=True)
img

In [ ]:
# Python
from rdkit import DataStructs
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Ensure plotly is available
try:
    import plotly.express as px
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go
except ImportError:
    !pip install plotly
    import plotly.express as px
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go

# Ensure TSNE and UMAP are available
from sklearn.manifold import TSNE

try:
    import umap
except ImportError:
    !pip install umap-learn
    import umap

# Keep valid fingerprints
df_fp = uniqueStructs.dropna(subset=["morgan"]).copy().reset_index(drop=True)
if df_fp.empty:
    raise ValueError("No valid Morgan fingerprints to cluster.")


# Convert RDKit ExplicitBitVect -> numpy array
def bv_to_array(bv):
    arr = np.zeros((bv.GetNumBits(),), dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(bv, arr)
    return arr


X = np.vstack([bv_to_array(fp) for fp in df_fp["morgan"]])

# Optional: dimensionality reduction before KMeans (helps speed/stability)
from sklearn.decomposition import PCA

pca_dims = min(50, X.shape[1], max(2, X.shape[0] - 1))
X_red = PCA(n_components=pca_dims, random_state=0).fit_transform(X)

# Choose k via silhouette (quick sweep)
candidate_k = range(2, 30)
best_k, best_score = None, -1
silh_scores = []
for k in candidate_k:
    if k >= len(df_fp):
        continue
    km = KMeans(n_clusters=k, n_init="auto", random_state=0)
    labels = km.fit_predict(X_red)
    score = silhouette_score(X_red, labels)
    silh_scores.append((k, score))
    if score > best_score:
        best_k, best_score = k, score

# Plot silhouette score vs k
if silh_scores:
    ks, scs = zip(*silh_scores)
    fig_k = go.Figure()
    fig_k.add_trace(go.Scatter(x=list(ks), y=list(scs), mode="lines+markers", name="silhouette"))
    fig_k.add_hline(y=best_score, line=dict(color="green", dash="dash"), annotation_text=f"best={best_score:.3f}")
    fig_k.add_vline(x=best_k, line=dict(color="red", dash="dot"), annotation_text=f"k={best_k}")
    fig_k.update_layout(template="plotly_white", title="Silhouette score vs number of clusters (k)",
                        xaxis_title="k", yaxis_title="silhouette score")
    fig_k.show()

# Final KMeans with chosen k
if best_k is None:
    best_k = 3
km_final = KMeans(n_clusters=best_k, n_init="auto", random_state=0)
df_fp["kmeans_cluster"] = km_final.fit_predict(X_red)

print(f"KMeans done. k={best_k}, silhouette={best_score:.3f}")

# 2D embeddings
emb_pca2 = PCA(n_components=2, random_state=0).fit_transform(X)
emb_tsne = TSNE(n_components=2, init="pca", learning_rate="auto",
                perplexity=min(30, max(5, len(df_fp) // 10)), random_state=0).fit_transform(X_red)
emb_umap = umap.UMAP(n_components=2, n_neighbors=min(15, max(2, len(df_fp) // 20)),
                     min_dist=0.1, metric="euclidean", random_state=0).fit_transform(X_red)

df_plot = pd.DataFrame({
    "PC1": emb_pca2[:, 0],
    "PC2": emb_pca2[:, 1],
    "TSNE1": emb_tsne[:, 0],
    "TSNE2": emb_tsne[:, 1],
    "UMAP1": emb_umap[:, 0],
    "UMAP2": emb_umap[:, 1],
    "cluster": df_fp["kmeans_cluster"].astype(str),
    "energy": df_fp.get("energy", pd.Series([np.nan] * len(df_fp)))
})

cluster_counts = df_fp["kmeans_cluster"].value_counts().sort_index()
avg_energy = df_fp.groupby("kmeans_cluster")["energy"].mean().reset_index()
avg_energy["kmeans_cluster"] = avg_energy["kmeans_cluster"].astype(str)

# Build 4x2 subplot figure
fig = make_subplots(
    rows=4, cols=2,
    subplot_titles=(
        f"Cluster counts (k={best_k})",
        "Average energy per cluster",
        "PCA colored by cluster",
        "PCA colored by energy",
        "t-SNE colored by cluster",
        "t-SNE colored by energy",
        "UMAP colored by cluster",
        "UMAP colored by energy"
    )
)

# Common color map
cluster_categories = sorted(df_plot["cluster"].unique(), key=lambda x: int(x) if x.isdigit() else x)
color_map = {c: px.colors.qualitative.T10[i % len(px.colors.qualitative.T10)] for i, c in enumerate(cluster_categories)}
colors = df_plot["cluster"].map(color_map)

# Row 1: Bars
fig.add_trace(
    go.Bar(x=cluster_counts.index.astype(str), y=cluster_counts.values, name="Counts"),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=avg_energy["kmeans_cluster"], y=avg_energy["energy"], name="Avg energy"),
    row=1, col=2
)

# Row 2: PCA
fig.add_trace(
    go.Scattergl(
        x=df_plot["PC1"], y=df_plot["PC2"],
        mode="markers",
        marker=dict(size=5, line=dict(width=0), color=colors),
        text=df_plot["cluster"],
        hovertemplate="PC1=%{x:.2f}<br>PC2=%{y:.2f}<br>cluster=%{text}<extra></extra>",
        showlegend=False
    ),
    row=2, col=1
)
fig.add_trace(
    go.Scattergl(
        x=df_plot["PC1"], y=df_plot["PC2"],
        mode="markers",
        marker=dict(size=5, line=dict(width=0), color=df_plot["energy"], colorscale="Viridis",
                    colorbar=dict(title="energy")),
        hovertemplate="PC1=%{x:.2f}<br>PC2=%{y:.2f}<br>energy=%{marker.color:.2f}<extra></extra>",
        showlegend=False
    ),
    row=2, col=2
)

# Row 3: t-SNE
fig.add_trace(
    go.Scattergl(
        x=df_plot["TSNE1"], y=df_plot["TSNE2"],
        mode="markers",
        marker=dict(size=5, line=dict(width=0), color=colors),
        text=df_plot["cluster"],
        hovertemplate="tSNE1=%{x:.2f}<br>tSNE2=%{y:.2f}<br>cluster=%{text}<extra></extra>",
        showlegend=False
    ),
    row=3, col=1
)
fig.add_trace(
    go.Scattergl(
        x=df_plot["TSNE1"], y=df_plot["TSNE2"],
        mode="markers",
        marker=dict(size=5, line=dict(width=0), color=df_plot["energy"], colorscale="Viridis",
                    colorbar=dict(title="energy")),
        hovertemplate="tSNE1=%{x:.2f}<br>tSNE2=%{y:.2f}<br>energy=%{marker.color:.2f}<extra></extra>",
        showlegend=False
    ),
    row=3, col=2
)

# Row 4: UMAP
fig.add_trace(
    go.Scattergl(
        x=df_plot["UMAP1"], y=df_plot["UMAP2"],
        mode="markers",
        marker=dict(size=5, line=dict(width=0), color=colors),
        text=df_plot["cluster"],
        hovertemplate="UMAP1=%{x:.2f}<br>UMAP2=%{y:.2f}<br>cluster=%{text}<extra></extra>",
        showlegend=False
    ),
    row=4, col=1
)
fig.add_trace(
    go.Scattergl(
        x=df_plot["UMAP1"], y=df_plot["UMAP2"],
        mode="markers",
        marker=dict(size=5, line=dict(width=0), color=df_plot["energy"], colorscale="Viridis",
                    colorbar=dict(title="energy")),
        hovertemplate="UMAP1=%{x:.2f}<br>UMAP2=%{y:.2f}<br>energy=%{marker.color:.2f}<extra></extra>",
        showlegend=False
    ),
    row=4, col=2
)

# Axes labels
fig.update_xaxes(title_text="Cluster", row=1, col=1)
fig.update_yaxes(title_text="Count", row=1, col=1)

fig.update_xaxes(title_text="Cluster", row=1, col=2)
fig.update_yaxes(title_text="Avg energy", row=1, col=2)

fig.update_xaxes(title_text="PC1", row=2, col=1)
fig.update_yaxes(title_text="PC2", row=2, col=1)
fig.update_xaxes(title_text="PC1", row=2, col=2)
fig.update_yaxes(title_text="PC2", row=2, col=2)

fig.update_xaxes(title_text="tSNE1", row=3, col=1)
fig.update_yaxes(title_text="tSNE2", row=3, col=1)
fig.update_xaxes(title_text="tSNE1", row=3, col=2)
fig.update_yaxes(title_text="tSNE2", row=3, col=2)

fig.update_xaxes(title_text="UMAP1", row=4, col=1)
fig.update_yaxes(title_text="UMAP2", row=4, col=1)
fig.update_xaxes(title_text="UMAP1", row=4, col=2)
fig.update_yaxes(title_text="UMAP2", row=4, col=2)

fig.update_layout(height=1600, width=1000, template="plotly_white",
                  title_text="Clustering overview (Bars, PCA, t-SNE, UMAP)")
fig.show()

# Merge cluster labels back to the original DataFrame by SMILES


Task: For each existing KMeans cluster (df_fp['kmeans_cluster']), compute the Maximum Common Substructure (MCS) across cluster members using RDKit, summarize SMARTS and size, and visualize representative substructures. Then, attach cluster labels back to outputResults by SMILES for downstream use.

In [ ]:
# Ensure RDKit MCS utilities are available
from rdkit.Chem import rdFMCS

# Prepare molecules per cluster
cluster_mols = (
    df_fp.dropna(subset=["mol", "kmeans_cluster"])
    .groupby("kmeans_cluster")["mol"]
    .apply(list)
    .to_dict()
)

# Compute MCS per cluster
mcs_results = []
mcs_queries = {}
for cid, mols in cluster_mols.items():
    mols_valid = [m for m in mols if m is not None and m.GetNumAtoms() > 0]
    if len(mols_valid) < 2:
        mcs_smarts, num_atoms, num_bonds = "", 0, 0
    else:
        res = rdFMCS.FindMCS(
            mols_valid,
            bondCompare=rdFMCS.BondCompare.CompareOrder,
            atomCompare=rdFMCS.AtomCompare.CompareElements,
            ringMatchesRingOnly=True,
            completeRingsOnly=False,
            matchValences=False,
            timeout=60
        )
        mcs_smarts = res.smartsString if res else ""
        num_atoms = res.numAtoms if res else 0
        num_bonds = res.numBonds if res else 0
    mcs_results.append(
        {"kmeans_cluster": cid, "MCS_SMARTS": mcs_smarts, "MCS_numAtoms": num_atoms, "MCS_numBonds": num_bonds})
    mcs_queries[cid] = Chem.MolFromSmarts(mcs_smarts) if mcs_smarts else None

mcs_df = pd.DataFrame(mcs_results).sort_values("kmeans_cluster").reset_index(drop=True)
mcs_df
#
# img = MolsToGridImage(
#     Chem.MolFromSmarts(mcs_df["MCS_SMARTS"]),
#     subImgSize=(400, 400),
#     molsPerRow=10,
#     maxMols=16,
#     legends=[f"Cluster:  {row['kmeans_cluster']}" for _, row in mcs_df.iterrows()],
#     useSVG=False,
#     returnPNG=True)
# img



In [ ]:
# Select 5 entries per cluster with lowest energy and rmsd, then display 15 molecules in one image

from rdkit.Chem.Draw import MolsToGridImage

if "df_fp" not in globals() or "mcs_df" not in globals():
    raise ValueError("Required data not found. Ensure clustering (df_fp) is computed.")

# For each cluster: pick top 5 by energy then by rmsd
top_per_cluster = []
for cid, g in df_fp.dropna(subset=["mol", "energy", "rmsd"]).groupby("kmeans_cluster"):
    g_sorted = g.sort_values(["energy", "rmsd"], ascending=[True, True]).head(5)
    top_per_cluster.append(g_sorted)

sel = pd.concat(top_per_cluster).reset_index(drop=True)

mols = sel["mol"].tolist()
legends = [f"cluster -  {cid} | E={e:.2f} | RMSD={r:.2f}" for cid, e, r in
           zip(sel["kmeans_cluster"], sel["energy"], sel["rmsd"])]

img = MolsToGridImage(
    mols,
    subImgSize=(400, 400),
    molsPerRow=5,
    legends=legends,
    useSVG=False,
    returnPNG=True
)

for i, row in sel.iterrows():
    Chem.MolToPDBFile(row['ROMol'], f"sel_{i}.pdb")
img

In [ ]:
# ... existing code ...
# Pure-RDKit contact table from ROMol vs receptor PDB (adds local_index passthrough if present)
# - inter_df columns: lig_index, lig_atom_idx, lig_x, lig_y, lig_z,
#                     prot_atom_idx, prot_atom_name, prot_resName, prot_chainID, prot_resSeq, distance,
#                     (plus any of: SMILES, filename, energy, rmsd, local_index if present in input df)
# - pdb_table: PDB text with one ATOM per interaction; occupancy holds distance (Å)

import os
import numpy as np
import pandas as pd


def _rdmol_has_3d(m: Chem.Mol) -> bool:
    return m is not None and m.GetNumConformers() > 0 and m.GetConformer().Is3D()


def _load_protein_atoms_from_pdb(pdb_path: str):
    """
    Parse PDB ATOM/HETATM records into a list of atoms with residue context and coordinates.
    """
    atoms = []
    if not os.path.isfile(pdb_path):
        raise FileNotFoundError(pdb_path)
    with open(pdb_path, "r") as fh:
        for line in fh:
            rec = line[0:6].strip()
            if rec not in ("ATOM", "HETATM"):
                continue
            try:
                x = float(line[30:38]);
                y = float(line[38:46]);
                z = float(line[46:54])
            except ValueError:
                continue
            name = line[12:16].strip()
            resName = line[17:20].strip()
            chainID = line[21:22].strip() or "A"
            try:
                resSeq = int(line[22:26].strip())
            except ValueError:
                resSeq = 0
            atoms.append({
                "idx": len(atoms),
                "name": name,
                "resName": resName,
                "chainID": chainID,
                "resSeq": resSeq,
                "x": x, "y": y, "z": z
            })
    if not atoms:
        raise ValueError("No atoms parsed from PDB")
    return atoms


def _compute_contacts(prot_atoms, lig_coords: np.ndarray, cutoff: float = 4.0):
    """
    Return list of (prot_idx, lig_idx, dist) for atom pairs within cutoff (Å).
    """
    contacts = []
    pcoords = np.array([[a["x"], a["y"], a["z"]] for a in prot_atoms], dtype=float)
    for li, l in enumerate(lig_coords):
        d = np.linalg.norm(pcoords - l, axis=1)
        within = np.where(d <= cutoff)[0]
        for pi in within:
            contacts.append((pi, li, float(d[pi])))
    return contacts


def interactions_to_pdb_table(inter_df: pd.DataFrame) -> str:
    """
    Build PDB-like ATOM lines, one per interaction.
    - Atom name: L<ligAtomIdx>
    - resName/chainID/resSeq: from protein atom's residue
    - Coordinates: ligand atom coordinates (x,y,z)
    - Occupancy field: stores distance
    """
    if inter_df is None or inter_df.empty:
        return ""
    lines = []
    serial = 1
    for _, r in inter_df.iterrows():
        x, y, z = r["lig_x"], r["lig_y"], r["lig_z"]
        atom_name = f"L{int(r['lig_atom_idx'])}"
        resName = str(r["prot_resName"])[:3].rjust(3)
        chainID = (str(r["prot_chainID"]) if pd.notna(r["prot_chainID"]) else "A")[:1]
        resSeq = int(r["prot_resSeq"])
        occ = float(r["distance"])
        line = (
            f"ATOM  {serial:5d} {atom_name:<4s} {resName} {chainID}{resSeq:4d}    "
            f"{x:8.3f}{y:8.3f}{z:8.3f}{occ:6.2f}{0.00:6.2f}          L"
        )
        lines.append(line)
        serial += 1
    return "\n".join(lines) + "\n"


def build_interaction_table_from_romol(df: pd.DataFrame, receptor_pdb: str,
                                       romol_col: str = "ROMol",
                                       cutoff: float = 4.0) -> tuple[pd.DataFrame, str]:
    """
    Compute residue–ligand atom contacts (distance cutoff) for each 3D ROMol row.

    Parameters
    - df: DataFrame with a ROMol column containing 3D conformers
    - receptor_pdb: path to receptor PDB file
    - romol_col: name of the column with RDKit Mol objects
    - cutoff: contact cutoff in Å

    Returns
    - inter_df: tidy DataFrame of contacts (includes local_index if present in df)
    - pdb_table: PDB-like table string for export
    """
    prot_atoms = _load_protein_atoms_from_pdb(receptor_pdb)

    out_rows = []
    id_cols = [c for c in ["SMILES", "filename", "energy", "rmsd", "local_index", "kmeans_cluster"] if c in df.columns]

    lig_counter = 0
    for _, row in df.iterrows():
        mol = row.get(romol_col, None)
        if not _rdmol_has_3d(mol):
            lig_counter += 1
            continue
        conf = mol.GetConformer()
        lig_coords = np.array([list(conf.GetAtomPosition(a.GetIdx())) for a in mol.GetAtoms()], dtype=float)
        contacts = _compute_contacts(prot_atoms, lig_coords, cutoff=cutoff)
        if not contacts:
            lig_counter += 1
            continue
        ids = {c: row[c] for c in id_cols}
        for (pi, li, dist) in contacts:
            pa = prot_atoms[pi]
            x, y, z = lig_coords[li]
            out_rows.append({
                "lig_index": lig_counter,
                "lig_atom_idx": li,
                "lig_x": x, "lig_y": y, "lig_z": z,
                "prot_atom_idx": pa["idx"],
                "prot_atom_name": pa["name"],
                "prot_resName": pa["resName"],
                "prot_chainID": pa["chainID"],
                "prot_resSeq": pa["resSeq"],
                "distance": dist,
                **ids
            })
        lig_counter += 1

    inter_df = pd.DataFrame(out_rows)
    pdb_table = interactions_to_pdb_table(inter_df)
    return inter_df, pdb_table


# ========= Example usage + Interaction heatmap =========
receptor_path = "../mols/Rad51/rad51Prepped.pdb"

# pick top 5 per cluster by lowest energy (ties by rmsd)
if "df_fp" in globals():
    top5_list = []
    for cid, g in df_fp.dropna(subset=["energy"]).groupby("kmeans_cluster"):
        gg = g.sort_values(["energy", "rmsd"], ascending=[True, True]).head(5)
        top5_list.append(gg)
    df_in = pd.concat(top5_list).reset_index(drop=True)
else:
    df_in = outputResults.head(15)

inter_df, pdb_tbl = build_interaction_table_from_romol(df_in, receptor_path, romol_col="ROMol", cutoff=4.0)
display(inter_df.head())


# # Derive simple interaction "type" from protein atom name
# def _infer_interaction_type(atom_name: str) -> str:
#     if pd.isna(atom_name):
#         return "other"
#     name = str(atom_name).upper()
#     if name.startswith("N"):
#         return "HBond/Polar"
#     if name.startswith("O"):
#         return "HBond/Polar"
#     if name.startswith("S"):
#         return "Polar"
#     if name.startswith("C"):
#         return "Hydrophobic"
#     return "Other"
#
#
# if not inter_df.empty:
#     inter_df["interaction_type"] = inter_df["prot_atom_name"].apply(_infer_interaction_type)
#     # label each ligand by cluster and rank within its cluster by energy
#     lig_meta = df_in.reset_index(drop=True)[["SMILES", "energy", "rmsd", "kmeans_cluster"]].copy()
#     lig_meta["lig_index"] = range(len(lig_meta))
#     inter_df = inter_df.merge(lig_meta, on="lig_index", how="left", validate="many_to_one")
#
#     # pivot: rows=candidate label (cluster-idx rank), cols=interaction_type, values=count
#     inter_df["candidate"] = inter_df.apply(
#         lambda r: f"C{int(r['kmeans_cluster'])}-L{int(r['lig_index'])} (E={r['energy']:.2f})", axis=1
#     )
#     heat_df = inter_df.groupby(["candidate", "interaction_type"]).size().unstack(fill_value=0)
#
#     # order candidates: by cluster then by energy asc
#     order_meta = inter_df.groupby("candidate")[["kmeans_cluster", "energy"]].first().sort_values(
#         by=["kmeans_cluster", "energy"], ascending=[True, True]
#     )
#     heat_df = heat_df.loc[order_meta.index]
#
#     # plot heatmap
#     import seaborn as sns
#     import matplotlib.pyplot as plt
#
#     plt.figure(figsize=(10, max(3, 0.35 * len(heat_df))))
#     sns.heatmap(heat_df, cmap="Blues", linewidths=0.5, linecolor="gray", cbar_kws={"label": "Contact count"})
#     plt.title("Interaction types for top 5 candidates per cluster (lowest energy)")
#     plt.xlabel("Interaction type")
#     plt.ylabel("Candidate (cluster-ligand, energy)")
#     plt.tight_layout()
#     plt.show()

# with open("ligand_protein_contacts.pdb","w") as f:
#     f.write(pdb_tbl)
# ... existing code ...


In [ ]:
# ... existing code ...
# ProLIF interaction fingerprints for selected ligands df_in
try:
    import prolif as plf
    import MDAnalysis as mda
except Exception as e:
    raise RuntimeError(
        "ProLIF/MDAnalysis are required in the current environment to compute interactions."
    ) from e

receptor_path = "../mols/Rad51/rad51Prepped.pdb"
u = mda.Universe(receptor_path, guess_bonds=True)
prot_plf = plf.Molecule.from_mda(u)

# Build list of RDKit mols from df_in (must have 3D conformers from ROMol)
pose_list = []
meta_rows = []
for i, row in df_in.reset_index(drop=True).iterrows():
    mol = row.get("ROMol", None)
    if mol is None or mol.GetNumConformers() == 0 or not mol.GetConformer().Is3D():
        continue
    pose_list.append(plf.Molecule.from_rdkit(mol))
    meta_rows.append({
        "lig_index": i,
        "SMILES": row.get("SMILES", None),
        "energy": row.get("energy", None),
        "rmsd": row.get("rmsd", None),
        "kmeans_cluster": row.get("kmeans_cluster", None)
    })

if not pose_list:
    raise ValueError("No 3D ligands found for ProLIF.")

fp = plf.Fingerprint(count=True)  # count occurrences per interaction
fp.run_from_iterable(pose_list, prot_plf)
idf = fp.to_dataframe()  # multiindex columns: (residue, interaction)
idf.index = range(len(idf))  # align with meta

meta_df = pd.DataFrame(meta_rows)
idf = pd.concat([meta_df, idf], axis=1)
idf
fp.plot_barcode(xlabel="Pose")
#
# # Collapse residue-specific columns into total counts per interaction type
# # Columns are MultiIndex: (ligand, residue, interaction); we sum over residues
# finger_cols = [c for c in idf.columns if isinstance(c, tuple) and len(c) == 3]
# if finger_cols:
#     collapsed = idf[finger_cols]
#     # sum across residues level (level=1)
#     collapsed = collapsed.groupby(level=2, axis=1).sum()
#     plot_df = pd.concat([idf[["lig_index", "SMILES", "energy", "rmsd", "kmeans_cluster"]], collapsed], axis=1)
# else:
#     plot_df = idf.copy()
#
# # Build a 2D interaction plot:
# # x-axis: energy; y-axis: total interaction count; color: dominant interaction type; symbol: cluster
# interaction_cols = [c for c in plot_df.columns if c not in ["lig_index", "SMILES", "energy", "rmsd", "kmeans_cluster"]]
# plot_df["total_interactions"] = plot_df[interaction_cols].sum(axis=1) if interaction_cols else 0
# if interaction_cols:
#     plot_df["dominant_interaction"] = plot_df[interaction_cols].idxmax(axis=1)
# else:
#     plot_df["dominant_interaction"] = "NA"
#
# # Ensure plotly is available
# try:
#     import plotly.express as px
# except ImportError:
#     !pip install plotly
#     import plotly.express as px
#
# fig = px.scatter(
#     plot_df,
#     x="energy",
#     y="total_interactions",
#     color="dominant_interaction",
#     symbol=plot_df["kmeans_cluster"].astype(str) if "kmeans_cluster" in plot_df.columns else None,
#     hover_data=["SMILES", "rmsd", "kmeans_cluster"] if "kmeans_cluster" in plot_df.columns else ["SMILES", "rmsd"],
#     title="2D Interaction Plot: Energy vs Total Interaction Count",
# )
# fig.update_traces(marker=dict(size=9, line=dict(width=0)))
# fig.update_layout(template="plotly_white", xaxis_title="Energy", yaxis_title="Total interaction count")
# fig.show()


In [ ]:
for i, pose_iterable in enumerate(pose_list):
    # display(fp.plot_lignetwork(pose_iterable, kind="frame"))
    view = fp.plot_3d(pose_iterable, prot_plf, frame=i, display_all=False)
    display(view)


In [ ]:
# Attach cluster labels to the main outputResults by SMILES
outputResults = outputResults.merge(
    df_fp.loc[:, ["SMILES", "kmeans_cluster"]],
    on="SMILES",
    how="left",
    validate="many_to_one"
)
outputResults

Notes:
- mcs_df contains per-cluster MCS SMARTS and sizes.
- The visualization highlights each cluster’s MCS on up to 16 representative molecules.
- Cluster labels are merged back into outputResults for further analysis.
